# Lab 01｜從問題到資料 Data Questions

<a href="https://colab.research.google.com/github/johnnychao/statistics-in-context-bilingual/blob/main/labs/colab/lab-01-data-questions.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

> Statistics in Context · Unit 1 · 原創合成資料 · 不評量 Python 語法


## Goal

情境：校務會議想知道「哪些晨間因素與學生到校時的學習準備度有關？」

- 找出 **observational unit（觀察單位）**、變數與變數型態。
- 將寬泛問題改寫成可用資料回答的 **investigative question（探究問題）**。
- 分清楚 identifier、categorical variable 與 quantitative variable。


## Setup

依序執行儲存格即可，不需要撰寫或背誦 Python。若想重新開始，請在 Colab 選擇 **Runtime → Restart session and run all**。

本 Lab 使用原創合成資料；所有代碼與數值均不對應真實學生。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [ ]:
# 集中設定：一般情況只需修改這一格的參數。
DATA_RELATIVE_PATH = "data/public/morning_routine_survey.csv"
REPO_RAW_BASE_URL = "https://raw.githubusercontent.com/johnnychao/statistics-in-context-bilingual/main"

LOCAL_REPO_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/content/statistics-in-context-bilingual"),
]


def load_repo_csv(relative_path):
    # 先找本機 repo，再讀 GitHub raw；失敗時提供繁中修復訊息。
    relative_path = Path(relative_path)
    for candidate_root in LOCAL_REPO_ROOT_CANDIDATES:
        candidate = candidate_root / relative_path
        if candidate.is_file():
            return pd.read_csv(candidate), str(candidate.resolve())

    remote_url = f"{REPO_RAW_BASE_URL}/{relative_path.as_posix()}"
    try:
        return pd.read_csv(remote_url), remote_url
    except Exception as exc:
        raise RuntimeError(
            "無法載入資料。請確認網路連線，或從 GitHub repo 根目錄執行此 Notebook。"
            f" 嘗試的遠端網址：{remote_url}。"
            " 若 repo 尚未發布，請先將 data/public 的 CSV 上傳至 main branch。"
        ) from exc


data, data_source = load_repo_csv(DATA_RELATIVE_PATH)
print(f"已載入 {len(data)} 筆資料｜Loaded {len(data)} rows")
print(f"來源 Source: {data_source}")


## Steps

### 1. 先看資料結構

每一列是一位合成調查學生；每一欄是一個變數。修改 `PREVIEW_ROWS`，觀察增加列數是否改變「觀察單位」的定義。


In [ ]:
# ✏️ 修改任務：把 6 改成 10，再執行此格。
PREVIEW_ROWS = 6

print(f"資料形狀 Data shape: {data.shape[0]} rows × {data.shape[1]} columns")
display(data.head(PREVIEW_ROWS))


### 2. 使用資料字典辨認變數

**Data dictionary（資料字典）**記錄欄位名稱、型態、單位、允許值與遺漏值規則。


In [ ]:
dictionary_path = "data/dictionaries/morning_routine_survey_dictionary.csv"
data_dictionary, dictionary_source = load_repo_csv(dictionary_path)
display(data_dictionary[[
    "column_name", "label_zh", "label_en", "data_type", "unit", "missing_value_rule"
]])


### 3. 聚焦可回答的問題

下面的欄位選擇只是起點。請改選至少一個類別變數與一個數值變數，再判斷問題是否可由這份資料回答。


In [ ]:
# ✏️ 修改任務：可改成 ate_breakfast、commute_mode、sleep_hours 等欄位。
SELECTED_COLUMNS = ["ate_breakfast", "sleep_hours", "readiness_score"]

unknown_columns = sorted(set(SELECTED_COLUMNS) - set(data.columns))
if unknown_columns:
    raise ValueError(f"欄位不存在 Unknown columns: {unknown_columns}")

display(data[SELECTED_COLUMNS].head(8))
print("可用筆數 Non-missing counts:")
display(data[SELECTED_COLUMNS].notna().sum().rename("non_missing_rows").to_frame())


<details>
<summary><strong>AP English Response frame</strong></summary>

> The observational unit is ____. The variable ____ is categorical/quantitative because ____. We can investigate whether ____ is associated with ____, but these observational data do not establish causation.

</details>

請用一句中文改寫探究問題，再用上方句型完成 2–3 句英文回答。


## Checks

執行自我檢核。全部通過表示資料與欄位設定可繼續使用。


In [ ]:
EXPECTED_COLUMNS = {
    "student_id", "grade", "commute_mode", "commute_minutes", "sleep_hours",
    "screen_minutes_after_9pm", "ate_breakfast", "arrival_status",
    "readiness_score", "response_source"
}
assert len(data) == 240, "資料應有 240 筆合成觀察值。"
assert EXPECTED_COLUMNS == set(data.columns), "欄位與資料字典不一致。"
assert data["student_id"].is_unique, "合成學生代碼應唯一。"
print("✅ Checks passed：240 個觀察單位、10 個欄位、識別碼唯一。")


## Next Steps

在 Lab 02，我們將用 frequency、relative frequency 與 bar chart 分析類別資料。請保留你的 investigative question，之後每張圖都要回到同一個校務情境解釋。
